In [1]:
from pydantic import BaseModel, Field
from typing import List, Dict, Union, Any
import json

class Parameters(BaseModel):
    selected_train_datasets: List[str]
    diff_dataset_usage: str
    siamese_nn_type: str
    learning_rate: float
    embedding_size: int
    use_butterworth_smoothed: bool

class MetricsBase(BaseModel):
    accuracy: float
    precision: float
    recall: float
    auroc: float
    rank_results: Dict[str, List[float]]

class Fold(MetricsBase):
    iteration: int
    stopped_at: int

class CombinedResults(MetricsBase):
    best_rank_1: List[Union[float, str]]

class ExperimentResult(BaseModel):
    parameters: Parameters
    folds: List[Fold]
    combined: CombinedResults
   
    def get_folds_above_accuracy(self, threshold: float) -> List[Fold]:
        """Returns all folds where accuracy is higher than threshold."""
        return [f for f in self.folds if f.accuracy > threshold]

    def get_best_fold_by_metric(self, metric_name: str = "accuracy") -> Fold:
        """Returns the fold with the highest value for a given metric."""
        return max(self.folds, key=lambda x: getattr(x, metric_name))


In [2]:
from pathlib import Path
from pydantic import ValidationError
import json

def load_all_experiments(directory_path: str) -> list[ExperimentResult]:
    folder = Path(directory_path)
    experiments = []

    for file_path in folder.glob("*.json"):
        try:
            with open(file_path, "r") as f:
                data = json.load(f)
                model_instance = ExperimentResult(**data)
                experiments.append(model_instance)
                print(f"Successfully loaded: {file_path.name}")
        
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"Failed to load {file_path.name}: {e}")
            
    return experiments

all_results = load_all_experiments("./json_reports")

Successfully loaded: 0_conv1d_1e-3_MIX_ALL_embd16__yolo_v26.json
Successfully loaded: 100_conv1d_1e-4_MIX_ALL_embd64__yolo_v26.json
Successfully loaded: 101_conv1d_1e-4_MIX_ALL_embd64__yolo_v26__butterworth.json
Successfully loaded: 102_conv1d_1e-4_MIX_ALL_embd128__yolo_v26.json
Successfully loaded: 103_conv1d_1e-4_MIX_ALL_embd128__yolo_v26__butterworth.json
Successfully loaded: 104_conv1d_1e-4_MIX_ALL_embd16__mocap.json
Successfully loaded: 105_conv1d_1e-4_MIX_ALL_embd16__mocap__butterworth.json
Successfully loaded: 106_conv1d_1e-4_MIX_ALL_embd32__mocap.json
Successfully loaded: 107_conv1d_1e-4_MIX_ALL_embd32__mocap__butterworth.json
Successfully loaded: 108_conv1d_1e-4_MIX_ALL_embd64__mocap.json
Successfully loaded: 109_conv1d_1e-4_MIX_ALL_embd64__mocap__butterworth.json
Successfully loaded: 10_conv1d_1e-3_MIX_ALL_embd32__mocap.json
Successfully loaded: 110_conv1d_1e-4_MIX_ALL_embd128__mocap.json
Successfully loaded: 111_conv1d_1e-4_MIX_ALL_embd128__mocap__butterworth.json
Successful

In [3]:
absolute_best = max(all_results, key=lambda x: x.combined.best_rank_1[0])

print(f"Parameters: {absolute_best.parameters}")
print(f"Overrall best rank-1: {absolute_best.combined.best_rank_1[0]:.4f}")

Parameters: selected_train_datasets=['mocap', 'yolo_v26', 'vitpose', 'openpose'] diff_dataset_usage='SKELETON_TYPE' siamese_nn_type='conv1d' learning_rate=0.0001 embedding_size=64 use_butterworth_smoothed=False
Overrall best rank-1: 97.0020


In [4]:
class ExperimentCollection(BaseModel):
    experiments: List[ExperimentResult]

    def filter(self, **kwargs) -> "ExperimentCollection":
        """
        Filters experiments based on parameter values.
        Usage: collection.filter(siamese_nn_type="conv1d", embedding_size=16)
        """
        filtered_list = self.experiments
        
        for key, value in kwargs.items():
            # We check if the key exists in the Parameters model
            filtered_list = [
                exp for exp in filtered_list 
                if getattr(exp.parameters, key, None) == value
            ]
            
        # Return a new collection object so you can chain filters
        return ExperimentCollection(experiments=filtered_list)
    
    def get_top_n(self, n: int = 3):
        """Returns the top N experiments sorted by combined accuracy."""
        sorted_list = sorted(self.experiments, key=lambda x: x.combined.best_rank_1[0], reverse=True)
        return [
            {
                'best rank-1': {
                    "value": experiment.combined.best_rank_1[0],
                    "classifier": experiment.combined.best_rank_1[1]
                },   
                'parameters': experiment.parameters.model_dump(),
                'early_stops_at': [exp_fold.stopped_at for exp_fold in experiment.folds],  
                'accuracy': experiment.combined.accuracy, 
                'precision': experiment.combined.precision,
                'recall': experiment.combined.recall,
                'auroc': experiment.combined.auroc,
            } 
            for experiment in sorted_list[:n]
        ]
        
    @property
    def count(self) -> int:
        return len(self.experiments)

collection = ExperimentCollection(experiments=all_results)
top_10 = collection.get_top_n(10)

In [5]:
for rank, value in enumerate(top_10):
    print(f"\n************* {rank+1} *************")
    pretty_value = json.dumps(value, indent=4)
    print(pretty_value)
    print("")


************* 1 *************
{
    "best rank-1": {
        "value": 97.00204377545998,
        "classifier": "SVM"
    },
    "parameters": {
        "selected_train_datasets": [
            "mocap",
            "yolo_v26",
            "vitpose",
            "openpose"
        ],
        "diff_dataset_usage": "SKELETON_TYPE",
        "siamese_nn_type": "conv1d",
        "learning_rate": 0.0001,
        "embedding_size": 64,
        "use_butterworth_smoothed": false
    },
    "early_stops_at": [
        8,
        7,
        6,
        6
    ],
    "accuracy": 0.8856209150326798,
    "precision": 0.8450844091360477,
    "recall": 0.6439651910707529,
    "auroc": 0.8608845329770498
}


************* 2 *************
{
    "best rank-1": {
        "value": 96.97940503432494,
        "classifier": "SVM"
    },
    "parameters": {
        "selected_train_datasets": [
            "yolo_v26"
        ],
        "diff_dataset_usage": "MIX_ALL",
        "siamese_nn_type": "conv1d",
        "l

In [6]:
transformer_collection = collection.filter(siamese_nn_type="transformer", embedding_size=32)
transformer_collection.count

27

In [7]:
transformer_collection = collection.filter(siamese_nn_type="transformer")
print(transformer_collection.count)
top_10_transformer = transformer_collection.get_top_n(10)

104


In [8]:
for rank, value in enumerate(top_10_transformer):
    print("Transformer model")
    print(f"\n************* {rank+1} *************")
    pretty_value = json.dumps(value, indent=4)
    print(pretty_value)
    print("")

Transformer model

************* 1 *************
{
    "best rank-1": {
        "value": 91.41653717571175,
        "classifier": "LR"
    },
    "parameters": {
        "selected_train_datasets": [
            "yolo_v26"
        ],
        "diff_dataset_usage": "MIX_ALL",
        "siamese_nn_type": "transformer",
        "learning_rate": 0.001,
        "embedding_size": 128,
        "use_butterworth_smoothed": true
    },
    "early_stops_at": [
        12,
        10,
        13,
        14
    ],
    "accuracy": 0.8808333333333334,
    "precision": 0.8205679862306369,
    "recall": 0.8857408267533674,
    "auroc": 0.9167393858880629
}

Transformer model

************* 2 *************
{
    "best rank-1": {
        "value": 90.82486531382145,
        "classifier": "LR"
    },
    "parameters": {
        "selected_train_datasets": [
            "mocap"
        ],
        "diff_dataset_usage": "MIX_ALL",
        "siamese_nn_type": "transformer",
        "learning_rate": 0.0001,
       

In [9]:
collection_32 = collection.filter(embedding_size=32)
collection_32.count
top_10_emb32 = collection_32.get_top_n(10)

for rank, value in enumerate(top_10_emb32):
    print("Embedding size 32\n")
    print(f"\n************* {rank+1} *************")
    pretty_value = json.dumps(value, indent=4)
    print(pretty_value)
    print("")

Embedding size 32


************* 1 *************
{
    "best rank-1": {
        "value": 96.24725108284092,
        "classifier": "SVM"
    },
    "parameters": {
        "selected_train_datasets": [
            "mocap",
            "yolo_v26",
            "vitpose",
            "openpose"
        ],
        "diff_dataset_usage": "SKELETON_TYPE",
        "siamese_nn_type": "conv1d",
        "learning_rate": 0.0001,
        "embedding_size": 32,
        "use_butterworth_smoothed": false
    },
    "early_stops_at": [
        6,
        6,
        6,
        7
    ],
    "accuracy": 0.9209150326797386,
    "precision": 0.7828894269572235,
    "recall": 0.7340143776012108,
    "auroc": 0.8633636864287464
}

Embedding size 32


************* 2 *************
{
    "best rank-1": {
        "value": 96.11422243633493,
        "classifier": "LR"
    },
    "parameters": {
        "selected_train_datasets": [
            "mocap",
            "yolo_v26",
            "vitpose",
            "open